<a href="https://colab.research.google.com/github/Hussam3d/Hussam3d/blob/main/Hussam_new_cylinder_new.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
# @title Tensile Test Dog-Bone Tube Generator
# @markdown Adjust the sliders below, then click Play to update the 3D view.

import fullcontrol as fc
import math
from google.colab import files

# @markdown ---
# @markdown ### 1. Actions & Style
preview_style = 'line' # @param ["line", "tube"]
download_gcode = False # @param {type:"boolean"}
design_name = 'dogbone_tensile_tube' # @param {type:"string"}

# @markdown ---
# @markdown ### 2. Grip Geometry (Both Ends)
tube_dia = 20 # @param {type:"slider", min:5, max:100, step:0.5}
grip_length = 10 # @param {type:"slider", min:1, max:50, step:0.5}
grip_EW = 0.8 # @param {"type":"slider","min":0.1,"max":2,"step":0.1}
grip_EH = 0.2 # @param {"type":"slider","min":0.05,"max":0.6,"step":0.1}

# @markdown ---
# @markdown ### 3. Main Body (Middle Section)
mid_dia = 10 # @param {type:"slider", min:2, max:100, step:0.5}
tube_length = 50 # @param {type:"slider", min:10, max:200, step:0.5}
body_EW = 0.4 # @param {type:"slider", min:0.1, max:2.0, step:0.01}
body_EH = 0.2 # @param {type:"slider", min:0.05, max:0.6, step:0.01}
segs_per_layer = 128 # @param {type:"slider", min:16, max:360, step:1}

# @markdown ---
# @markdown ### 4. Printer Parameters
printer_name = 'prusa_i3' # @param ['generic', 'ultimaker2plus', 'prusa_i3', 'ender_3', 'cr_10', 'bambulab_x1', 'toolchanger_T0']
nozzle_temp = 210 # @param {type:"slider", min:180, max:280, step:1}
bed_temp = 55 # @param {type:"slider", min:0, max:110, step:1}
print_speed = 1000 # @param {type:"slider", min:100, max:4000, step:100}
fan_percent = 100 # @param {type:"slider", min:0, max:100, step:1}

# --- Geometry Setup ---
initial_z = grip_EH * 0.4
radius_end = tube_dia / 2.0
radius_mid = mid_dia / 2.0

safe_grip_length = min(grip_length, (tube_length / 2.0) - 0.5)

steps = []
steps.append(fc.Extruder(on=False))
steps.append(fc.Point(x=radius_end, y=0, z=initial_z))
steps.append(fc.Extruder(on=True))

z = initial_z
angle = 0

# --- Generate Continuous Toolpath ---
while z <= tube_length:
    in_grip = (z <= safe_grip_length) or (z >= tube_length - safe_grip_length)
    current_EH = grip_EH if in_grip else body_EH

    if in_grip:
        r = radius_end
    else:
        # Cosine interpolation for flawless geometric fillet
        u = (z - safe_grip_length) / (tube_length - 2 * safe_grip_length)
        r = radius_mid + (radius_end - radius_mid) * ((1 + math.cos(2 * math.pi * u)) / 2)

    x = r * math.cos(angle)
    y = r * math.sin(angle)
    steps.append(fc.Point(x=x, y=y, z=z))

    dz = current_EH / segs_per_layer
    d_angle = (math.pi * 2) / segs_per_layer

    z += dz
    angle += d_angle

# --- 3D Plotting ---
if preview_style == 'tube':
    plot_controls = fc.PlotControls(style='tube', zoom=0.7, initialization_data={'extrusion_width': body_EW, 'extrusion_height': body_EH})
else:
    plot_controls = fc.PlotControls(style='line', zoom=0.7)

fc.transform(steps, 'plot', plot_controls)

# --- Downloading & Post-Processing ---
if download_gcode:
    print(f"Preparing {design_name}.gcode for download...")
    gcode_controls = fc.GcodeControls(
        printer_name=printer_name,
        initialization_data={
            'primer': 'front_lines_then_y',
            'print_speed': print_speed,
            'nozzle_temp': nozzle_temp,
            'bed_temp': bed_temp,
            'fan_percent': fan_percent,
            'extrusion_width': body_EW,
            'extrusion_height': body_EH
        }
    )

    # Generate base gcode
    raw_gcode = fc.transform(steps, 'gcode', gcode_controls)

    # Post-Process to inject M221 for thicker grips
    new_gcode = []
    current_flow = 100
    target_grip_flow = int((grip_EW / body_EW) * 100)

    for line in raw_gcode.split('\n'):
        if line.startswith('G1') or line.startswith('G0'):
            parts = line.split()
            for p in parts:
                if p.startswith('Z'):
                    try:
                        z_val = float(p[1:])
                        # Add a tiny 0.05 buffer for floating point precision
                        if z_val <= (safe_grip_length + 0.05):
                            desired_flow = target_grip_flow
                        elif z_val >= (tube_length - safe_grip_length - 0.05):
                            desired_flow = target_grip_flow
                        else:
                            desired_flow = 100

                        if desired_flow != current_flow:
                            new_gcode.append(f'M221 S{desired_flow} ; Dynamic Flow Override')
                            current_flow = desired_flow
                    except ValueError:
                        pass
        new_gcode.append(line)

    new_gcode.append('M221 S100 ; Reset printer flow to 100% at end')
    final_gcode = '\n'.join(new_gcode)

    with open(f'{design_name}.gcode', 'w') as f:
        f.write(final_gcode)

    files.download(f'{design_name}.gcode')
    print("Download started!")